# History matching z parameters with `pestpp-ies` and the `pypestutils` fieldgen helpers

This notebook uses `generate_2d_grid_field_from_z()` in a model forward run.  Hydraulic conductivity (hk) is not parameterized with multipliers.  Instead, every model cell has a "z" parameter, and the hk field is a deterministic function of the z values:

$$\log_{10}(hk) = \log_{10}(\bar{hk}) + \text{spatial average of } z$$

The spatial averaging (and so the spatial correlation of hk) is controlled by a variogram range, bearing and anisotropy ratio.  These vary in space too: each is interpolated from a sparse set of "hyper parameter" pilot points, and the pilot point values are also adjustable parameters.  So the parameters are:

* one z value per model cell. A priori these are independent, with mean zero and variance one, so the prior covariance matrix of the z parameters is the identity matrix
* hyper parameter pilot points for the hk variogram range, bearing and anisotropy

The model is the `synth2d` problem from the PEST++ benchmarks: one layer, specified inflow along the right edge flowing to a river on the left edge, and three pumping wells that switch on after a steady-state period.  Observations are groundwater levels at four monitoring wells and the river flux.  The "truth" is a realization from the prior, and `pestpp-ies` is used to condition an ensemble to noisy observations of it.

In [ ]:
import os
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

This notebook uses `flopy`, `pyEMU`, `ipywidgets`, `modflow6` and `pestpp-ies`.  Let's make sure we can import those deps:

In [ ]:
import flopy
import ipywidgets
import ppu_helpers
import pyemu
from IPython.display import display

from pypestutils import helpers as ppu

In [ ]:
bin_dir = "synth2d_bin"
if os.path.exists(bin_dir):
    shutil.rmtree(bin_dir)
os.makedirs(bin_dir)
flopy.utils.get_modflow(bin_dir, downloads_dir=bin_dir)
pyemu.utils.get_pestpp(bin_dir, downloads_dir=bin_dir, release_id="5.2.27")

## The model

Build and run the model.  The functions that build the model and that are used in the forward run are in `ppu_helpers.py`.  Note that hk is written to an external file, `hk.txt`, which the forward run replaces each time it runs.

In [ ]:
mf6_exe = [f for f in os.listdir(bin_dir) if f.startswith("mf6")][0]
model_ws = "synth2d_model"
sim = ppu_helpers.build_synth2d_model(
    model_ws, exe_name=os.path.abspath(os.path.join(bin_dir, mf6_exe))
)
gwf = sim.get_model()
nrow, ncol = gwf.dis.nrow.data, gwf.dis.ncol.data
xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
nrow, ncol

In [ ]:
hds = flopy.utils.HeadFile(os.path.join(model_ws, "synth2d.hds"))
heads = hds.get_data(kstpkper=(0, 12))[0]
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.set_aspect("equal")
cb = ax.pcolormesh(xc, yc, heads)
plt.colorbar(cb, label="head (m)")
for nm, (_, i, j) in ppu_helpers.synth2d_head_obs_cells(nrow, ncol).items():
    ax.scatter(xc[i, j], yc[i, j], c="k", marker="^")
    ax.text(xc[i, j], yc[i, j], " " + nm)
for _, i, j in ppu_helpers.synth2d_well_cells(nrow, ncol):
    ax.scatter(xc[i, j], yc[i, j], c="r", marker="o")
_ = ax.set_title(
    "heads at end of simulation; monitoring wells (black), pumping wells (red)",
    loc="left",
    fontsize=8,
)

The helpers read the grid from the MODFLOW-6 binary grid file.  Keep a copy of it, so it is still there if a model run fails part way through:

In [ ]:
grb_fname = os.path.join(model_ws, "org.grb")
shutil.copy2(os.path.join(model_ws, "synth2d.dis.grb"), grb_fname)

## The hk field and its hyper parameters

The hk field is controlled by:
* the mean hk (the field when all z values are zero) and the variance of log10(hk). These are fixed.
* the hk variogram range, bearing and anisotropy ratio at each cell.  These are interpolated from hyper parameter pilot points.  Bearing is measured in degrees clockwise from north.

Each hyper parameter field has a prior mean, a prior standard deviation (for range and anisotropy, the standard deviation is of log10 of the value), and a variogram range (the spatial correlation of its pilot point values).  The means, standard deviations and variogram ranges below are the starting values for the sliders further down.

The parameter bounds follow the pyemu convention: the bounds span `sigma_range` prior standard deviations, so they are the prior mean ± `sigma_range` / 2 standard deviations (in log10 space for range and anisotropy).  With `sigma_range` = 4, the bounds are ± 2 standard deviations:

In [ ]:
hk_mean, hk_variance = 5.0, 0.25
sigma_range = 4.0
hyper_prior = pd.DataFrame(
    {
        "mean": [800.0, 45.0, 3.0],
        "sd": [0.4, 45.0, 0.5],
        "corrlen": [1250.0, 1500.0, 1750.0],
    },
    index=list(ppu_helpers.HYPER_FIELDS.keys()),
)
hyper_prior["transform"] = [ppu_helpers.HYPER_FIELDS[n] for n in hyper_prior.index]


def add_bounds(prior):
    """set the "lower_bound" and "upper_bound" columns to the prior mean
    +/- sigma_range / 2 prior standard deviations (in log10 space for
    log-transformed parameters)"""
    half_width = sigma_range / 2.0 * prior["sd"]
    log = prior["transform"] == "log"
    prior["lower_bound"] = np.where(
        log, prior["mean"] * 10.0**-half_width, prior["mean"] - half_width
    )
    prior["upper_bound"] = np.where(
        log, prior["mean"] * 10.0**half_width, prior["mean"] + half_width
    )
    return prior


hyper_prior = add_bounds(hyper_prior)
hyper_prior

Each hyper parameter field is interpolated from its pilot points with an isotropic variogram (which also sets the spatial correlation of the hyper parameter pilot point prior).  The sliders below set:
* the hyper parameter pilot point spacing (in cells)
* for each hyper parameter field: the prior mean, the prior standard deviation and the variogram range

The plots show one prior realization of each hyper parameter field and the hk field that results from one (fixed) set of z values.  The pilot point locations are the black dots.

The values the sliders are left at are used to set up the PEST interface below, so adjust them before running the rest of the notebook.

In [ ]:
from pypestutils.logger import temporary_logger_level


def prior_from_settings(settings):
    """hyper_prior with the prior means, standard deviations and variogram
    ranges from the sliders, and the bounds that follow from them"""
    prior = hyper_prior.copy()
    for name in prior.index:
        prior.loc[name, "mean"] = settings[f"{name}_mean"]
        prior.loc[name, "sd"] = settings[f"{name}_sd"]
        prior.loc[name, "corrlen"] = settings[f"{name}_corrlen"]
    return add_bounds(prior)


def make_hyper_ppdfs(settings, prior):
    """pilot point dataframes for each hyper parameter field, with the prior
    mean as "value" and that field's (isotropic) variogram range as "corrlen\""""
    base = ppu.get_2d_pp_info_structured_grid(settings["pp_space"], grb_fname)
    ppdfs = {}
    for name in prior.index:
        df = base.loc[:, ["ppname", "x", "y", "zone"]].copy()
        df["value"] = prior.loc[name, "mean"]
        df["corrlen"] = settings[f"{name}_corrlen"]
        df["bearing"] = 0.0
        df["aniso"] = 1.0
        ppdfs[name] = df
    return ppdfs


def hyper_cov(df, sd, corrlen):
    """pyemu covariance matrix of one hyper parameter field's pilot point values
    (in log10 space for log-transformed fields), indexed by pilot point name"""
    vario = pyemu.geostats.ExpVario(contribution=sd**2, a=corrlen)
    gs = pyemu.geostats.GeoStruct(variograms=vario)
    return gs.covariance_matrix(df.x, df.y, names=df.ppname.tolist())


def draw_hyper_ppdfs(ppdfs, num_reals, rng, prior):
    """draw prior hyper parameter pilot point values, clipped to the bounds"""
    draws = {}
    for name, df in ppdfs.items():
        p = prior.loc[name]
        cov = hyper_cov(df, p["sd"], df.corrlen.iloc[0])
        vals = rng.multivariate_normal(np.zeros(df.shape[0]), cov.x, num_reals)
        if p["transform"] == "log":
            vals = p["mean"] * 10.0**vals
        else:
            vals = p["mean"] + vals
        draws[name] = np.clip(vals, p["lower_bound"], p["upper_bound"])
    return draws


z_preview = np.random.default_rng(1).standard_normal(nrow * ncol)


def preview(**settings):
    # silence the pypestutils log messages while previewing
    with temporary_logger_level("WARNING"):
        prior = prior_from_settings(settings)
        ppdfs = make_hyper_ppdfs(settings, prior)
        draws = draw_hyper_ppdfs(ppdfs, 1, np.random.default_rng(2), prior)
        for name, df in ppdfs.items():
            df["value"] = draws[name][0]
        hk, fields = ppu_helpers.hk_from_z(
            z_preview, ppdfs, grb_fname, hk_mean, hk_variance
        )
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    for ax, (name, arr) in zip(
        axes, list(fields.items()) + [("log10 hk", np.log10(hk))]
    ):
        ax.set_aspect("equal")
        cb = ax.pcolormesh(xc, yc, arr)
        plt.colorbar(cb, ax=ax, shrink=0.8)
        ax.scatter(ppdfs["range"].x, ppdfs["range"].y, c="k", s=8)
        ax.set_title(name, loc="left")
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def slider(cls, value, vmin, vmax, step, desc):
    return cls(
        value=value,
        min=vmin,
        max=vmax,
        step=step,
        description=desc,
        continuous_update=False,
        # fixed label width so the slider bars are a usable length and line up
        style={"description_width": "230px"},
        layout=ipywidgets.Layout(width="520px"),
    )


sliders = {
    "pp_space": slider(ipywidgets.IntSlider, 9, 3, 12, 1, "hyper pp spacing (cells)")
}
# slider limits for the prior mean and standard deviation of each hyper parameter
# field: (min, max, step).  The standard deviation must be greater than zero,
# since it sets the width of the parameter bounds
mean_limits = {
    "range": (200.0, 3200.0, 50.0),
    "bearing": (-45.0, 135.0, 5.0),
    "aniso": (1.0, 10.0, 0.25),
}
sd_limits = {
    "range": (0.01, 0.5, 0.01),
    "bearing": (1.0, 45.0, 1.0),
    "aniso": (0.01, 0.5, 0.01),
}
for name, p in hyper_prior.iterrows():
    sd_desc = "log10 sd" if p["transform"] == "log" else "sd"
    sliders[f"{name}_mean"] = slider(
        ipywidgets.FloatSlider, p["mean"], *mean_limits[name], f"{name} field: mean"
    )
    sliders[f"{name}_sd"] = slider(
        ipywidgets.FloatSlider, p["sd"], *sd_limits[name], f"{name} field: {sd_desc}"
    )
    sliders[f"{name}_corrlen"] = slider(
        ipywidgets.FloatSlider,
        p["corrlen"],
        250.0,
        10000.0,
        250.0,
        f"{name} field: variogram range (m)",
    )

# two columns of sliders, filled top to bottom, then the preview plots
names = list(sliders)
half = (len(names) + 1) // 2
columns = ipywidgets.HBox(
    [
        ipywidgets.VBox([sliders[n] for n in names[:half]]),
        ipywidgets.VBox(
            [sliders[n] for n in names[half:]],
            layout=ipywidgets.Layout(margin="0 0 0 40px"),
        ),
    ]
)
display(columns, ipywidgets.interactive_output(preview, sliders))

These are the settings that will be used:

In [ ]:
settings = {name: s.value for name, s in sliders.items()}
hyper_prior = prior_from_settings(settings)
settings

## Parameter files for the forward run

Now write the files that `pestpp-ies` will put parameter values into:
* `z.csv`: one z value per cell, with the (zero-based) node number. The initial values are zero, which gives hk equal to the mean everywhere
* `hyperpp_range.csv`, `hyperpp_bearing.csv` and `hyperpp_aniso.csv`: the hyper parameter pilot points, with the prior mean as the initial value
* `hk_info.csv`: the (fixed) hk mean and variance

In [ ]:
hyper_ppdfs = make_hyper_ppdfs(settings, hyper_prior)

for name, df in hyper_ppdfs.items():
    df.to_csv(os.path.join(model_ws, f"hyperpp_{name}.csv"), index=False)
zdf = pd.DataFrame({"node": np.arange(nrow * ncol), "z": 0.0})
zdf.to_csv(os.path.join(model_ws, "z.csv"), index=False)
pd.Series({"mean": hk_mean, "variance": hk_variance}, name="value").to_csv(
    os.path.join(model_ws, "hk_info.csv")
)
shutil.copy2("ppu_helpers.py", os.path.join(model_ws, "ppu_helpers.py"))
hyper_ppdfs["range"]

The forward run calls `ppu_helpers.apply_hk_from_z()` before MODFLOW-6 runs.  It reads the parameter files, interpolates the hyper parameter fields from their pilot points, then calls `generate_2d_grid_field_from_z()` with the z values to make `hk.txt`:

In [ ]:
import inspect

print(inspect.getsource(ppu_helpers.hk_from_z))
print(inspect.getsource(ppu_helpers.apply_hk_from_z))

## The PEST interface

Use `PstFrom` to set up the interface.  The z values and the hyper parameter pilot point values are "direct" parameters: `pestpp-ies` writes their values straight into the files above.

In [ ]:
template_ws = "synth2d_z_template"
pf = pyemu.utils.PstFrom(
    original_d=model_ws,
    new_d=template_ws,
    remove_existing=True,
    longnames=True,
    spatial_reference=gwf.modelgrid,
    zero_based=True,
    start_datetime="1-1-2020",
)

The z parameters.  Their prior standard deviation is one, so their bounds are ± `sigma_range` / 2:

In [ ]:
pf.add_parameters(
    "z.csv",
    par_type="grid",
    index_cols=["node"],
    use_cols=["z"],
    par_style="direct",
    transform="none",
    par_name_base="z",
    pargp="z",
    lower_bound=-sigma_range / 2.0,
    upper_bound=sigma_range / 2.0,
)

The hyper parameter pilot points:

In [ ]:
for name, p in hyper_prior.iterrows():
    pf.add_parameters(
        f"hyperpp_{name}.csv",
        par_type="grid",
        index_cols=["ppname"],
        use_cols=["value"],
        par_style="direct",
        transform=p["transform"],
        par_name_base=f"hyper{name}",
        pargp=f"hyper{name}",
        lower_bound=p["lower_bound"],
        upper_bound=p["upper_bound"],
    )

Add the forward run command, and run it once in the template directory to make the files it writes.  This also checks it works:

In [ ]:
pf.extra_py_imports.append("ppu_helpers")
pf.pre_py_cmds.append("ppu_helpers.apply_hk_from_z()")

cwd = os.getcwd()
os.chdir(template_ws)
try:
    ppu_helpers.apply_hk_from_z()
finally:
    os.chdir(cwd)

## Observations

Groundwater levels and river flux, plus the hk field and the hyper parameter fields as zero-weight observations, so we can see what happens to them:

In [ ]:
hcols = [c.upper() for c in ppu_helpers.synth2d_head_obs_cells(nrow, ncol)]
pf.add_observations(
    "head_obs.csv",
    insfile="head_obs.csv.ins",
    index_cols="time",
    use_cols=hcols,
    prefix="gwlevel",
    obsgp="gwlevel",
)
pf.add_observations(
    "riv_obs.csv",
    insfile="riv_obs.csv.ins",
    index_cols="time",
    use_cols=["RIVFLUX"],
    prefix="rivflux",
    obsgp="rivflux",
)
for name in ["hk"] + [f"hyper_{n}" for n in hyper_prior.index]:
    pf.add_observations(
        f"{name}.txt", prefix=name.replace("_", ""), obsgp=name.replace("_", "")
    )

In [ ]:
pf.mod_sys_cmds.append("mf6")
pst = pf.build_pst(os.path.join(template_ws, "synth2d.pst"), version=2)
pst.npar, pst.nobs

In [ ]:
_ = [print(line.rstrip()) for line in open(os.path.join(template_ws, "forward_run.py"))]

## The prior parameter ensemble

The z parameters are independent standard normal values.  The hyper parameter pilot point values are spatially correlated, using the variogram of each hyper parameter field (set with the sliders above).  We draw one extra realization to use as the "truth":

In [ ]:
# use a smaller ensemble (and fewer iterations, below) when running in CI
in_ci = os.environ.get("CI", "").lower() == "true"
num_reals = 30 if in_ci else 100
par = pst.parameter_data

zpar = par.loc[par.pargp == "z", :].copy()
zpar["node"] = zpar.node.astype(int)
zpar = zpar.sort_values("node")

# prior covariance matrix: unit variance and no correlation for the z
# parameters, and the geostatistical covariance of each hyper parameter field
# (in log10 space for log-transformed fields)
blocks = [
    pyemu.Cov(
        x=np.ones((zpar.shape[0], 1)), names=zpar.parnme.tolist(), isdiagonal=True
    )
]
for name, df in hyper_ppdfs.items():
    p = hyper_prior.loc[name]
    hcov = hyper_cov(df, p["sd"], settings[f"{name}_corrlen"])
    # rename the pilot points to their parameter names
    hpar = par.loc[par.pargp == f"hyper{name}", :].set_index("ppname").parnme
    blocks.append(pyemu.Cov(x=hcov.x, names=hpar.loc[hcov.row_names].tolist()))
names = [n for b in blocks for n in b.row_names]
x = np.zeros((len(names), len(names)))
start = 0
for b in blocks:
    end = start + b.shape[0]
    x[start:end, start:end] = b.as_2d
    start = end
cov = pyemu.Cov(x=x, names=names)

pst.add_parbnd_center()
pst.parameter_data["org_parval1"] = pst.parameter_data.parval1.copy()
pst.parameter_data["parval1"] = pst.parameter_data.bnd_center
pe = pyemu.ParameterEnsemble.from_gaussian_draw(
    pst, cov=cov, num_reals=num_reals + 1, rng=np.random.default_rng(4269)
)
pst.parameter_data["parval1"] = pst.parameter_data.pop("org_parval1")
pe.enforce()
df = pe._df.copy()
df.index = ["truth"] + list(range(num_reals))
truth_pars = df.loc["truth", :].copy()
pe = pyemu.ParameterEnsemble(pst=pst, df=df.loc[df.index != "truth", :])
pe.to_binary(os.path.join(template_ws, "prior.jcb"))
pe.shape

Let's look at a few prior hk realizations.  We can make them directly with the same function the forward run uses:

In [ ]:
def fields_from_pars(pars):
    """hk and the hyper parameter fields for one parameter realization"""
    z = pars.loc[zpar.parnme].values
    ppdfs = {}
    for name, df in hyper_ppdfs.items():
        df = df.copy()
        hpar = par.loc[par.pargp == f"hyper{name}", :].set_index("ppname")
        df["value"] = pars.loc[hpar.loc[df.ppname, "parnme"]].values
        ppdfs[name] = df
    return ppu_helpers.hk_from_z(z, ppdfs, grb_fname, hk_mean, hk_variance)


fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for ax, real in zip(axes.flatten(), pe.index[:8]):
    hk, _ = fields_from_pars(pe.loc[real, :])
    ax.set_aspect("equal")
    cb = ax.pcolormesh(xc, yc, np.log10(hk), vmin=-0.5, vmax=2.0)
    ax.set_title(f"log10 hk, realization {real}", loc="left", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

## The truth and the observations

Copy the executables (MODFLOW-6 and `pestpp-ies`) into the template directory, so the forward run does not rely on them being on the PATH.  Then run the model with the truth parameters, and set the observation values to the truth outputs plus noise (0.1 m for groundwater levels, 1% for river flux).  The hk and hyper parameter field observations keep zero weight:

In [ ]:
for f in os.listdir(bin_dir):
    shutil.copy2(os.path.join(bin_dir, f), os.path.join(template_ws, f))

truth_ws = "synth2d_z_truth"
if os.path.exists(truth_ws):
    shutil.rmtree(truth_ws)
shutil.copytree(template_ws, truth_ws)
org_parval1 = pst.parameter_data.parval1.copy()
pst.parameter_data.loc[truth_pars.index, "parval1"] = truth_pars.values
pst.write_input_files(pst_path=truth_ws)
pst.parameter_data.loc[:, "parval1"] = org_parval1
pyemu.os_utils.run("python forward_run.py", cwd=truth_ws)

In [ ]:
obs = pst.observation_data
obs.loc[:, "weight"] = 0.0
truth_vals = {}
nz_rng = np.random.default_rng(4)
for oname, fname in [("gwlevel", "head_obs.csv"), ("rivflux", "riv_obs.csv")]:
    tdf = pd.read_csv(os.path.join(truth_ws, fname), index_col=0)
    tdf.columns = [c.lower() for c in tdf.columns]
    oobs = obs.loc[obs.oname == oname, :]
    for nme, row in oobs.iterrows():
        val = tdf.loc[float(row.time), row.usecol]
        sigma = 0.5 if oname == "gwlevel" else max(0.1 * abs(val), 1.0)
        truth_vals[nme] = val
        obs.loc[nme, "obsval"] = val  # + nz_rng.normal(0.0, sigma)
        if "rivflux" not in oname:
            obs.loc[nme, "weight"] = 1.0 / sigma
            # obs.loc[nme, "standard_deviation"] = sigma
truth_hk = np.loadtxt(os.path.join(truth_ws, "hk.txt"))
truth_hyper = {
    n: np.loadtxt(os.path.join(truth_ws, f"hyper_{n}.txt")) for n in hyper_prior.index
}
pst.nnz_obs

## Run `pestpp-ies`

Write the control file:

In [ ]:
pst.pestpp_options["ies_par_en"] = "prior.jcb"
pst.pestpp_options["ies_num_reals"] = num_reals
pst.pestpp_options["overdue_giveup_fac"] = 10.0
# pst.pestpp_options["ies_multimodal_alpha"] = 1.0
pst.control_data.noptmax = 3 if in_ci else 7
pst.write(os.path.join(template_ws, "synth2d.pst"), version=2)

In [ ]:
master_ws = "synth2d_z_master"
num_workers = min(10, os.cpu_count())
pyemu.os_utils.start_workers(
    template_ws,
    "pestpp-ies",
    "synth2d.pst",
    num_workers=num_workers,
    worker_root=".",
    master_dir=master_ws,
)

## Results

The objective function:

In [ ]:
phi = pd.read_csv(os.path.join(master_ws, "synth2d.phi.meas.csv"), index_col=0)
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
ax.plot(phi.index, phi.iloc[:, 6:].values, c="0.5", lw=0.5)
ax.plot(phi.index, phi["mean"], c="b", lw=2)
ax.hlines(pst.nnz_obs, *ax.get_xlim(), "r", ls="--")
ax.set_yscale("log")
ax.set_xlabel("iteration")
_ = ax.set_ylabel("phi")


Prior (grey) and posterior (blue) groundwater levels and river flux, with the observations (red):

In [ ]:
last = int(phi.index.max())
pr_oe = pd.read_csv(os.path.join(master_ws, "synth2d.0.obs.csv"), index_col=0)
pt_oe = pd.read_csv(os.path.join(master_ws, f"synth2d.{last}.obs.csv"), index_col=0)
groups = [
    ("gwlevel", u, ".")
    for u in sorted(obs.loc[obs.oname == "gwlevel", "usecol"].unique())
] + [("rivflux", "rivflux", "")]
fig, axes = plt.subplots(1, len(groups), figsize=(16, 3.2))
for ax, (oname, usecol, m) in zip(axes, groups):
    oobs = obs.loc[(obs.oname == oname) & (obs.usecol == usecol), :].copy()
    oobs["time"] = oobs.time.astype(float)
    oobs = oobs.sort_values("time")
    ax.plot(oobs.time, pr_oe.loc[:, oobs.obsnme].T.values, c="0.5", lw=0.5)
    ax.plot(oobs.time, pt_oe.loc[:, oobs.obsnme].T.values, c="b", lw=0.5)
    ax.scatter(oobs.time, oobs.obsval, marker=m, c="r", s=10, zorder=10)
    ax.set_title(usecol, loc="left")
plt.tight_layout()

The truth, the prior and posterior ensemble mean of log10 hk and of the hyper parameter fields, and the posterior standard deviation of log10 hk:

In [ ]:
def obs_field(oe, group, log=False):
    oobs = obs.loc[obs.obgnme == group, :]
    arr = oe.loc[:, oobs.obsnme].values
    if log:
        arr = np.log10(arr)
    fields = np.zeros((oe.shape[0], nrow, ncol))
    fields[:, oobs.i.astype(int).values, oobs.j.astype(int).values] = arr
    return fields


rows = [("log10 hk", "hk", True, np.log10(truth_hk))] + [
    (n, f"hyper{n}", False, truth_hyper[n]) for n in hyper_prior.index
]
fig, axes = plt.subplots(len(rows), 4, figsize=(14, 3.2 * len(rows)))
for ax_row, (label, group, log, truth) in zip(axes, rows):
    pr = obs_field(pr_oe, group, log)
    pt = obs_field(pt_oe, group, log)
    vmin, vmax = truth.min(), truth.max()
    for ax, arr, title, kw in zip(
        ax_row,
        [truth, pr.mean(axis=0), pt.mean(axis=0), pt.std(axis=0)],
        ["truth", "prior mean", "posterior mean", "posterior std"],
        [dict(vmin=vmin, vmax=vmax)] * 3 + [{}],
    ):
        ax.set_aspect("equal")
        cb = ax.pcolormesh(xc, yc, arr, **kw)
        plt.colorbar(cb, ax=ax, shrink=0.8)
        ax.set_title(f"{label}: {title}", loc="left", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()

Prior (grey) and posterior (blue) distributions of the mean of each hyper parameter field, with the truth (red):

In [ ]:
type(pt_oe)

In [ ]:
def obs_field(oe, real, group="hk", log=False):
    oobs = obs.loc[obs.obgnme == group, :]
    arr = oe.loc[real, oobs.obsnme].values
    if log:
        arr = np.log10(arr)
    fields = np.zeros((nrow, ncol))
    fields[oobs.i.astype(int).values, oobs.j.astype(int).values] = arr
    return fields


rows = pt_oe.index[:4]

fig, axes = plt.subplots(len(rows), 3, figsize=(14, 3.2 * len(rows)))
for ax_row, real in zip(axes, rows):
    pr = obs_field(pr_oe, real, "hk", log)
    pt = obs_field(pt_oe, real, "hk", log)
    vmin, vmax = truth.min(), truth.max()
    for ax, arr, label in zip(
        ax_row.flatten(), [truth_hk, pr, pt], ["truth", "prior", "post"]
    ):
        ax.set_aspect("equal")
        cb = ax.pcolormesh(xc, yc, np.log10(arr), **kw)
        plt.colorbar(cb, ax=ax, shrink=0.8)
        ax.set_title(f"{label}", loc="left", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, len(hyper_prior.index), figsize=(12, 3))
for ax, name in zip(axes, hyper_prior.index):
    oobs = obs.loc[obs.obgnme == f"hyper{name}", "obsnme"]
    ax.hist(pr_oe.loc[:, oobs].mean(axis=1), color="0.5", alpha=0.5, density=True)
    ax.hist(pt_oe.loc[:, oobs].mean(axis=1), color="b", alpha=0.5, density=True)
    ax.axvline(truth_hyper[name].mean(), c="r", lw=2)
    ax.set_title(f"mean {name}", loc="left")
plt.tight_layout()